# 🏥 Mask2Former Component Ablation: Full Standard Mask2Former (Baseline Control — 0.68 Benchmark)
### Dedicated Kaggle GPU Runner — EXPERIMENT_2 (Set A Suite)
**Ablation Mode:** `baseline` | **Output Archive:** `EXPERIMENT_2_RESULTS_RUN_0_BASELINE.zip`

---

### Configuration Overview
- **Ablation Mode:** `baseline` (Pretrained Swin-Tiny Backbone + MSDeformAttn Multi-Scale Pixel Decoder + Masked Cross-Attention + Query Self-Attention.)
- **Model Base:** `facebook/mask2former-swin-tiny-ade-semantic` (Authentic Pretrained Swin-Tiny Mask2Former)
- **RGB-Only Standard Pipeline:** Pure 3-channel input `(3, 1024, 1024)`. Zero depth dependency.
- **Patient 32 4K Canvas Bug Fix:** Dynamically extracts `imageHeight`/`imageWidth` from JSON labels to prevent coordinate truncation.
- **Standardized Line Thickness:** Uses thickness `35` on raw canvas matching EXPERIMENT_1 bit-for-bit.
- **Dynamic Dataset Discovery:** Automatically scans `/kaggle/input` using scored directory matching.
- **Deep Supervision Hungarian Loss:** Evaluated natively at every decoder layer.
- **Multi-Query Semantic Post-Processing:** Cooperative multi-query ensemble projection over all 100 queries.
- **Automated Packaging:** Packages all outputs into `{zip_name}` directly in `/kaggle/working/` for one-click download.


## Step 1: Environment Diagnostics & Package Installation
Verify CUDA acceleration, memory properties, and ensure required packages (`transformers`, `surface-distance`) are available.


In [ ]:
import os
import sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Install required dependencies if missing
!pip install -q transformers surface-distance medpy > /dev/null 2>&1 || true

import numpy as np
# Monkeypatch legacy NumPy aliases removed in NumPy 2.0 (for surface_distance/medpy)
for attr, val in [('Inf', np.inf), ('Infinity', np.inf), ('NAN', np.nan), ('NaN', np.nan)]:
    if not hasattr(np, attr):
        setattr(np, attr, val)
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
if not hasattr(np, 'float_'):
    np.float_ = np.float64

import torch
print('=' * 70)
print('🚀 GPU & ENVIRONMENT DIAGNOSTICS')
print('=' * 70)
print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('⚠️ WARNING: GPU is not detected! Enable GPU Accelerator in Kaggle sidebar (Settings -> Accelerator -> GPU).')


## Step 2: Automatic Scored Dataset Discovery
Scans `/kaggle/input` using scored heuristics to locate `Train`, `Val`, and `Test` image splits.


In [ ]:
import glob
from pathlib import Path

def find_dataset_split(split_keyword):
    candidates = []
    search_roots = ['/kaggle/input', '/kaggle/working', './data', '../data']
    for search_root in search_roots:
        if not os.path.exists(search_root):
            continue
        for root, dirs, _ in os.walk(search_root, followlinks=True):
            parts_lower = [p.lower() for p in Path(root).parts]
            if split_keyword.lower() in parts_lower and 'images' in parts_lower:
                score = 0
                if 'khoatrytopublish' in parts_lower:
                    score += 50
                if 'l3d' in parts_lower or any('l3d' in p for p in parts_lower):
                    score += 30
                if 'laparoscopic' in parts_lower:
                    score += 20
                candidates.append((score, root))
            elif os.path.basename(root).lower() == split_keyword.lower():
                if 'images' in [d.lower() for d in dirs]:
                    img_dir = os.path.join(root, 'images')
                    candidates.append((10, img_dir))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

train_img_dir = find_dataset_split('train')
val_img_dir = find_dataset_split('val')
test_img_dir = find_dataset_split('test')

print('=' * 70)
print('📂 DATASET DISCOVERY RESULTS')
print('=' * 70)
print(f'Train Images: {train_img_dir}')
print(f'Val Images  : {val_img_dir}')
print(f'Test Images : {test_img_dir}')
print('=' * 70)


## Step 3: Standardized L3D Dataset Loader
- Preserves dynamic canvas resolution from JSON labels to prevent Patient 32 4K truncation.
- Standardizes line thickness to 35.
- Employs RGB-only standard inputs `(1024, 1024, 3)` with discrete 2D ground truth maps `(1024, 1024)`.


In [ ]:
import cv2
import json
import numpy as np
from torch.utils.data import Dataset, DataLoader

class L3DSurgicalDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        valid_exts = {'.png', '.jpg', '.jpeg', '.bmp'}
        self.img_paths = sorted([
            os.path.join(img_dir, f) for f in os.listdir(img_dir)
            if Path(f).suffix.lower() in valid_exts
        ])
        print(f'Loaded {len(self.img_paths)} frames from {img_dir}')

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        rgb_img = self.load_image(img_path)
        gt_2d = self.load_mask_2d(img_path)
        return rgb_img, gt_2d, str(img_path)

    @staticmethod
    def load_image(path):
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if img.shape[0] != 1024 or img.shape[1] != 1024:
            img = cv2.resize(img, (1024, 1024), interpolation=cv2.INTER_LINEAR)
        return img

    @staticmethod
    def load_mask_2d(path):
        path_str = str(path)
        json_path = os.path.splitext(path_str)[0] + '.json'
        if not os.path.exists(json_path):
            parent = os.path.dirname(path_str)
            base_stem = os.path.splitext(os.path.basename(path_str))[0]
            candidates = [
                os.path.join(parent, '..', 'labels', base_stem + '.json'),
                os.path.join(parent, 'labels', base_stem + '.json'),
                os.path.join(parent.replace('images', 'labels'), base_stem + '.json')
            ]
            for c in candidates:
                if os.path.exists(c):
                    json_path = c
                    break

        if not os.path.exists(json_path):
            return np.zeros((1024, 1024), dtype=np.int32)

        with open(json_path, 'r') as f:
            data = json.load(f)

        # Dynamic canvas size extraction (Patient 32 4K canvas bug fix)
        img_h = data.get('imageHeight', 1080)
        img_w = data.get('imageWidth', 1920)
        canvas = np.zeros((img_h, img_w), dtype=np.uint8)

        for shape in data.get('shapes', []):
            label = str(shape.get('label', '')).lower()
            if label.startswith('r') or 'ridge' in label or 'rigde' in label:
                color = 1
            elif label.startswith('s') or 'sil' in label:
                color = 2
            elif label.startswith('l') or 'lig' in label or 'falc' in label:
                color = 3
            else:
                color = 0

            if color > 0:
                points = shape.get('points', [])
                for i in range(1, len(points)):
                    pt1 = tuple(map(int, points[i - 1]))
                    pt2 = tuple(map(int, points[i]))
                    cv2.line(canvas, pt1, pt2, color, 35)

        if canvas.shape[0] != 1024 or canvas.shape[1] != 1024:
            canvas = cv2.resize(canvas, (1024, 1024), interpolation=cv2.INTER_NEAREST)

        return canvas.astype(np.int32)

def collate_fn_l3d(batch):
    images = [item[0] for item in batch]
    masks = [item[1] for item in batch]
    paths = [item[2] for item in batch]
    return images, masks, paths


## Step 4: Metric Evaluation (Macro Dice, IoU, ASSD, Patient 40)
Standardized TopoNet evaluation functions matching EXPERIMENT_1.


In [ ]:
import numpy as np
# Ensure NumPy 2.0 compatibility for legacy surface_distance
for attr, val in [('Inf', np.inf), ('Infinity', np.inf), ('NAN', np.nan), ('NaN', np.nan)]:
    if not hasattr(np, attr):
        setattr(np, attr, val)

try:
    import surface_distance
    from surface_distance import metrics as sd_metrics
    HAS_SURFACE_DIST = True
except Exception:
    HAS_SURFACE_DIST = False

def evaluation(pred, gt):
    smooth = 1e-5
    intersection = np.sum(pred * gt)
    dice = (2.0 * intersection + smooth) / (np.sum(pred) + np.sum(gt) + smooth)
    iou = dice / (2.0 - dice)
    return float(iou), float(dice)

def compute_toponet_metrics(pred_map, gt_2d):
    pred_channels = np.array([pred_map == i for i in range(4)]).astype(np.uint8)
    gt_channels = np.array([gt_2d == i for i in range(4)]).astype(np.uint8)

    # Macro foreground metric (Classes 1, 2, 3)
    iou, dice = evaluation(pred_channels[1:].flatten(), gt_channels[1:].flatten())

    # Per-class Dice
    class_dices = {}
    class_names = {1: 'ridge', 2: 'silhouette', 3: 'falciform'}
    for c, name in class_names.items():
        _, c_dice = evaluation(pred_channels[c].flatten(), gt_channels[c].flatten())
        class_dices[name] = float(c_dice)

    assd = None
    if HAS_SURFACE_DIST:
        try:
            if np.count_nonzero(pred_channels[1:]) == 0:
                assd = 80.0
            else:
                temp_assd = []
                for i in range(3):
                    gt_c = np.array(gt_channels[i + 1], dtype=bool)
                    pred_c = np.array(pred_channels[i + 1], dtype=bool)
                    if not gt_c.any() or not pred_c.any():
                        temp_assd.append(80.0)
                        continue
                    sd = sd_metrics.compute_surface_distances(gt_c, pred_c, (1.0, 1.0))
                    avg_sd = surface_distance.compute_average_surface_distance(sd)
                    val = avg_sd[1]
                    temp_assd.append(val if not np.isnan(val) and val < 500.0 else 80.0)
                mean_dist = float(np.mean(temp_assd)) if temp_assd else 80.0
                assd = mean_dist if mean_dist < 500.0 else 80.0
        except Exception:
            assd = 80.0

    return dice, iou, class_dices, assd


## Step 5: Model Initialization (`facebook/mask2former-swin-tiny-ade-semantic`) & Architectural Ablation
Configured for: **Full Standard Mask2Former (Baseline Control — 0.68 Benchmark)**


In [ ]:
from transformers import (
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'facebook/mask2former-swin-tiny-ade-semantic'
ABLATION_TYPE = 'baseline'

print(f'Loading AutoImageProcessor from {MODEL_NAME}...')
processor = AutoImageProcessor.from_pretrained(MODEL_NAME, reduce_labels=False, ignore_index=255)

print(f'Loading Mask2FormerForUniversalSegmentation from {MODEL_NAME}...')
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    ignore_mismatched_sizes=True
).to(device)

# ==================== ARCHITECTURAL ABLATION PATCHING ====================
if ABLATION_TYPE == 'wo_masked_attn':
    # Ablation 1: Disable Masked Cross-Attention (Set attn_mask = None -> Full Global Attention)
    for l in model.model.transformer_module.decoder.layers:
        orig = l.forward_pre
        def patch_fn(fn):
            def patched(*args, **kwargs):
                kwargs['encoder_attention_mask'] = None
                return fn(*args, **kwargs)
            return patched
        l.forward_pre = patch_fn(orig)
        l.forward = patch_fn(l.forward)
    print('⚠️ Masked Attention DISABLED: All 9 decoder layers operating in FULL GLOBAL CROSS-ATTENTION mode.')

elif ABLATION_TYPE == 'wo_multiscale':
    # Ablation 2: Disable Multi-Scale Feature Cycling (Lock all layers to single stride-16 feature level)
    orig_tm_forward = model.model.transformer_module.forward
    def single_scale_tm_forward(multi_scale_features, mask_features, output_hidden_states=False, output_attentions=False):
        single_feat = multi_scale_features[1]  # Lock to stride 16 (24x24)
        single_multi_scale = [single_feat, single_feat, single_feat]
        return orig_tm_forward(single_multi_scale, mask_features, output_hidden_states=output_hidden_states, output_attentions=output_attentions)
    model.model.transformer_module.forward = single_scale_tm_forward
    print('⚠️ Multi-Scale Feature Cycling DISABLED: All 9 decoder layers locked to single stride-16 level.')

elif ABLATION_TYPE == 'wo_self_attn':
    # Ablation 3: Disable Query Self-Attention (Bypass self_attn -> Independent Parallel Queries)
    for l in model.model.transformer_module.decoder.layers:
        def patch_self_attn():
            def patched(*args, **kwargs):
                hs = kwargs.get('hidden_states', args[0] if len(args) > 0 else None)
                return torch.zeros_like(hs), None
            return patched
        l.self_attn.forward = patch_self_attn()
    print('⚠️ Query Self-Attention DISABLED: Queries operating as independent parallel detectors (no inter-query communication).')

else:
    print('✅ Full Standard Mask2Former Active: Masked Cross-Attention + Multi-Scale Feature Cycling + Query Self-Attention.')


## Step 6: Model Training with Native Deep Supervision Hungarian Loss
- Uses native multi-layer Hungarian loss (`outputs.loss`).
- Uses multi-query semantic ensemble projection (`processor.post_process_semantic_segmentation`).
- Runs 60 epochs with Cosine Annealing scheduler and AMP FP16 mixed precision.


In [ ]:
import time
from tqdm.auto import tqdm

EPOCHS = 60
BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
LEARNING_RATE = 8e-5
WEIGHT_DECAY = 3e-5
SAVE_DIR = '/kaggle/working/results_mask2former_baseline'
os.makedirs(SAVE_DIR, exist_ok=True)

train_dataset = L3DSurgicalDataset(train_img_dir)
val_dataset = L3DSurgicalDataset(val_img_dir)
test_dataset = L3DSurgicalDataset(test_img_dir) if test_img_dir else None

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_l3d)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_l3d)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

best_val_dice = 0.0
best_ckpt_path = os.path.join(SAVE_DIR, 'best_baseline.pth')
latest_ckpt_path = os.path.join(SAVE_DIR, 'latest_baseline.pth')
log_file = os.path.join(SAVE_DIR, 'training_metrics.json')
metrics_history = []

print('=' * 75)
print('🚀 STARTING TRAINING: Full Standard Mask2Former (Baseline Control — 0.68 Benchmark)')
print(f'   Train Samples: {len(train_dataset)} | Val Samples: {len(val_dataset)}')
print(f'   Effective Batch Size: {BATCH_SIZE * ACCUMULATION_STEPS} | Total Epochs: {EPOCHS}')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Epoch {epoch:02d}/{EPOCHS} [Train]')
    for step, (batch_rgb, batch_gt, _) in pbar:
        inputs = processor(images=batch_rgb, segmentation_maps=batch_gt, return_tensors='pt')
        pixel_values = inputs['pixel_values'].to(device)
        mask_labels = [m.to(device) for m in inputs['mask_labels']]
        class_labels = [c.to(device) for c in inputs['class_labels']]

        if scaler is not None:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
                loss = outputs.loss / ACCUMULATION_STEPS
            scaler.scale(loss).backward()
        else:
            outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
            loss = outputs.loss / ACCUMULATION_STEPS
            loss.backward()

        total_train_loss += outputs.loss.item() * len(batch_rgb)

        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{outputs.loss.item():.4f}'})

    scheduler.step()
    avg_train_loss = total_train_loss / len(train_dataset)
    epoch_time = time.time() - epoch_start

    # ==================== VALIDATION ====================
    model.eval()
    val_losses = []
    val_dices, val_ious, val_assds = [], [], []
    class_dices_accum = {'ridge': [], 'silhouette': [], 'falciform': []}
    patient_40_dices = []

    with torch.no_grad():
        for batch_rgb, batch_gt, batch_paths in tqdm(val_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Val]'):
            inputs = processor(images=batch_rgb, segmentation_maps=batch_gt, return_tensors='pt')
            pixel_values = inputs['pixel_values'].to(device)
            mask_labels = [m.to(device) for m in inputs['mask_labels']]
            class_labels = [c.to(device) for c in inputs['class_labels']]

            if scaler is not None:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
            else:
                outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)

            val_losses.append(outputs.loss.item() * len(batch_rgb))

            target_sizes = [(1024, 1024)] * len(batch_rgb)
            pred_maps = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)

            for p_tensor, gt_arr, p_path in zip(pred_maps, batch_gt, batch_paths):
                p_arr = p_tensor.cpu().numpy()
                d, iou, c_dices, assd = compute_toponet_metrics(p_arr, gt_arr)
                val_dices.append(d)
                val_ious.append(iou)
                if assd is not None:
                    val_assds.append(assd)
                for k in class_dices_accum:
                    class_dices_accum[k].append(c_dices[k])
                if 'patient40' in p_path.lower() or 'p40' in p_path.lower():
                    patient_40_dices.append(d)

    mean_val_loss = np.sum(val_losses) / len(val_dataset)
    mean_val_dice = float(np.mean(val_dices))
    mean_val_iou = float(np.mean(val_ious))
    mean_val_assd = float(np.mean(val_assds)) if len(val_assds) > 0 else 0.0
    mean_ridge = float(np.mean(class_dices_accum['ridge']))
    mean_sil = float(np.mean(class_dices_accum['silhouette']))
    mean_falc = float(np.mean(class_dices_accum['falciform']))
    mean_p40 = float(np.mean(patient_40_dices)) if len(patient_40_dices) > 0 else 0.0

    print(
        f'👉 Epoch {epoch:02d} ({epoch_time:.1f}s) | '
        f'Tr Loss: {avg_train_loss:.4f} | Val Loss: {mean_val_loss:.4f} | '
        f'Val Dice: {mean_val_dice:.4f} (Ridge: {mean_ridge:.4f}, Sil: {mean_sil:.4f}, Falc: {mean_falc:.4f}) | '
        f'P40 Dice: {mean_p40:.4f}'
    )

    record = {
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'val_loss': mean_val_loss,
        'val_macro_dice': mean_val_dice,
        'val_macro_iou': mean_val_iou,
        'val_assd': mean_val_assd,
        'val_ridge_dice': mean_ridge,
        'val_silhouette_dice': mean_sil,
        'val_falciform_dice': mean_falc,
        'val_patient40_dice': mean_p40,
        'epoch_duration_sec': epoch_time
    }
    metrics_history.append(record)

    with open(log_file, 'w') as f:
        json.dump(metrics_history, f, indent=2)

    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_val_dice': best_val_dice,
            'mode': 'baseline'
        }, best_ckpt_path)
        print(f'  🏆 New Best Model! Val Dice: {best_val_dice:.4f} -> {best_ckpt_path}')

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'best_val_dice': best_val_dice,
        'mode': 'baseline'
    }, latest_ckpt_path)


## Step 7: Final Test Evaluation & One-Click Zip Packaging
Evaluates the best checkpoint on the unseen Test split (109 frames) and bundles all outputs into a single downloadable zip file.


In [ ]:
import shutil

if test_dataset is not None and os.path.exists(best_ckpt_path):
    print('=' * 75)
    print(f'🔬 RUNNING FINAL TEST EVALUATION USING BEST CHECKPOINT (Val Dice: {best_val_dice:.4f})')
    print('=' * 75)

    best_ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_l3d)
    test_dices, test_ious, test_assds = [], [], []
    test_class_dices = {'ridge': [], 'silhouette': [], 'falciform': []}

    with torch.no_grad():
        for batch_rgb, batch_gt, _ in tqdm(test_loader, desc='Testing'):
            inputs = processor(images=batch_rgb, segmentation_maps=batch_gt, return_tensors='pt')
            pixel_values = inputs['pixel_values'].to(device)
            if scaler is not None:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(pixel_values=pixel_values)
            else:
                outputs = model(pixel_values=pixel_values)

            target_sizes = [(1024, 1024)] * len(batch_rgb)
            pred_maps = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)

            for p_tensor, gt_arr in zip(pred_maps, batch_gt):
                p_arr = p_tensor.cpu().numpy()
                d, iou, c_dices, assd = compute_toponet_metrics(p_arr, gt_arr)
                test_dices.append(d)
                test_ious.append(iou)
                if assd is not None:
                    test_assds.append(assd)
                for k in test_class_dices:
                    test_class_dices[k].append(c_dices[k])

    final_summary = {
        'mode': 'baseline',
        'best_val_dice': best_val_dice,
        'test_macro_dice': float(np.mean(test_dices)),
        'test_macro_iou': float(np.mean(test_ious)),
        'test_assd': float(np.mean(test_assds)) if len(test_assds) > 0 else 0.0,
        'test_ridge_dice': float(np.mean(test_class_dices['ridge'])),
        'test_silhouette_dice': float(np.mean(test_class_dices['silhouette'])),
        'test_falciform_dice': float(np.mean(test_class_dices['falciform']))
    }

    summary_file = os.path.join(SAVE_DIR, 'final_summary.json')
    with open(summary_file, 'w') as f:
        json.dump(final_summary, f, indent=2)

    print('=' * 75)
    print(f'🏁 FINAL TEST RESULTS [baseline]:')
    print(f'   Best Val Dice      : {final_summary["best_val_dice"]:.4f}')
    print(f'   Test Macro Dice    : {final_summary["test_macro_dice"]:.4f}')
    print(f'   Test Macro IoU     : {final_summary["test_macro_iou"]:.4f}')
    print(f'   Test ASSD (pixels) : {final_summary["test_assd"]:.4f}')
    print(f'   Ridge Dice         : {final_summary["test_ridge_dice"]:.4f}')
    print(f'   Silhouette Dice    : {final_summary["test_silhouette_dice"]:.4f}')
    print(f'   Falciform Dice     : {final_summary["test_falciform_dice"]:.4f}')
    print('=' * 75)

# ==================== ZIP PACKAGING ====================
zip_base = '/kaggle/working/EXPERIMENT_2_RESULTS_RUN_0_BASELINE'
print(f'📦 Packaging results into {zip_base}.zip...')
shutil.make_archive(zip_base, 'zip', SAVE_DIR)
print(f'🎉 Packaging Complete! Download: /kaggle/working/EXPERIMENT_2_RESULTS_RUN_0_BASELINE.zip')
